In [33]:
#!pip install roboflow


from roboflow import Roboflow
rf = Roboflow(api_key="hRTUI0MqBME6xajesmw7")
project = rf.workspace("cam-ctadw").project("makinegormesi-6bypv")
version = project.version(2)
dataset = version.download("coco")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to MakineGormesi-2 in coco:: 100%|██████████| 1551/1551 [00:00<00:00, 5432.37it/s]


In [3]:
CLASSES= ['cam', 'kagit', 'metal', 'pil', 'plastik']

In [41]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pycocotools.coco import COCO
from torchvision import transforms

CLASSES = ['cam', 'kagit', 'metal', 'pil', 'plastik'] # kendi sınıfların
NUM_CLASSES = len(CLASSES) + 1  # + background
IMG_SIZE = 512
BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = "/content/MakineGormesi-2"  # COCO JSON dosyalarının olduğu klasör

class COCODataset(Dataset):
    def __init__(self, img_dir, json_file, transform=None):
        self.img_dir = img_dir
        self.coco = COCO(json_file)
        self.ids = list(self.coco.imgs.keys())
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        path = os.path.join(self.img_dir, img_info['file_name'])
        image = Image.open(path).convert("RGB")

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)

        boxes = []
        labels = []

        for ann in anns:
            xmin, ymin, w, h = ann['bbox']
            xmax = xmin + w
            ymax = ymin + h
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(ann['category_id'])  # COCO category_id 1'den başlıyor

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64)
        }

        if self.transform:
            image = self.transform(image)

        return image, target

# Transform ve DataLoader
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

train_dataset = COCODataset(
    img_dir=os.path.join(DATA_DIR, "train"),
    json_file=os.path.join(DATA_DIR, "train/_annotations.coco.json"),
    transform=transform
)

val_dataset = COCODataset(
    img_dir=os.path.join(DATA_DIR, "valid"),
    json_file=os.path.join(DATA_DIR, "valid/_annotations.coco.json"),
    transform=transform
)

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


In [48]:
import torchvision
import torch
import torch.optim as optim
from tqdm import tqdm
import os

# Cihaz ayarı (kodda eksikse diye ekledim)
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
# NUM_CLASSES tanımlı olmalı

# Model ve optimizer
model = torchvision.models.detection.ssdlite320_mobilenet_v3_large(num_classes=NUM_CLASSES).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
EPOCHS = 50

for epoch in range(EPOCHS):
    # --- TRAINING ---
    model.train()
    train_loss = 0
    # Train loader'ı tqdm ile sarmalıyoruz
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for images, targets in loop:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

        # Progress bar'da anlık loss göstermek için (opsiyonel)
        loop.set_postfix(loss=loss.item())

    # --- VALIDATION ---
    # DİKKAT: Validation loss hesaplamak için model.train() modunda kalmalı
    # Torchvision detection modelleri eval modunda loss döndürmez!
    # Ancak gradyan hesaplamayı kapatıyoruz (no_grad).

    val_loss = 0
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            # Model train modunda olduğu için loss_dict döner
            # (model.eval() çağrılmadı veya tekrar model.train() yapıldı)
            loss_dict = model(images, targets)
            val_loss += sum(loss for loss in loss_dict.values()).item()

    # Ortalamaları hesapla
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1} | Avg Train Loss: {avg_train_loss:.4f} | Avg Val Loss: {avg_val_loss:.4f}")

# Model kaydet
os.makedirs("saved_models", exist_ok=True)
torch.save(model.state_dict(), "saved_models/ssd_mobilenet_coco.pth")
print("✅ Eğitim tamamlandı! Model kaydedildi.")

Epoch 1/50: 100%|██████████| 78/78 [00:22<00:00,  3.47it/s, loss=8.86]


Epoch 1 | Avg Train Loss: 10.8352 | Avg Val Loss: 9.0282


Epoch 2/50: 100%|██████████| 78/78 [00:21<00:00,  3.60it/s, loss=7.9]


Epoch 2 | Avg Train Loss: 8.0537 | Avg Val Loss: 7.4968


Epoch 3/50: 100%|██████████| 78/78 [00:21<00:00,  3.59it/s, loss=6.6]


Epoch 3 | Avg Train Loss: 6.7637 | Avg Val Loss: 6.6969


... (90 outputs left out)

Epoch 49/50: 100%|██████████| 78/78 [00:21<00:00,  3.56it/s, loss=2.46]


Epoch 49 | Avg Train Loss: 0.9422 | Avg Val Loss: 5.3811


Epoch 50/50: 100%|██████████| 78/78 [00:22<00:00,  3.50it/s, loss=5.02]


Epoch 50 | Avg Train Loss: 0.9383 | Avg Val Loss: 5.4408
✅ Eğitim tamamlandı! Model kaydedildi.


In [53]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from torchvision.ops import box_iou
import os

# --- 1. TEST VERİ SETİNİ VE LOADER'I AYARLAMA ---

# Test verilerinin yolu (Senin belirttiğin klasör)
TEST_DIR = "/content/MakineGormesi-2/test"
TEST_JSON = os.path.join(TEST_DIR, "_annotations.coco.json")

# Eğitimde kullandığın transform aynen kullanılmalı (Resize 512x512)
# Yukarıdaki kodunda 'transform' değişkeni zaten tanımlı, onu kullanıyoruz.
test_dataset = COCODataset(
    img_dir=TEST_DIR,
    json_file=TEST_JSON,
    transform=transform  # Senin kodundaki transform değişkeni
)

# Test Loader
test_loader = DataLoader(
    test_dataset,
    batch_size=4,       # Bellek durumuna göre artırabilirsin (örn: 8 veya 16)
    shuffle=False,      # Testte karıştırma yapılmaz
    collate_fn=collate_fn # Senin kodundaki collate_fn
)

# --- 2. MODELİ DEĞERLENDİRME (EVALUATION) ---

# Modeli değerlendirme moduna al
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

all_preds = []
all_targets = []

print(f"📂 '{TEST_DIR}' klasöründeki {len(test_dataset)} görüntü test ediliyor...")

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="Test İlerlemesi"):
        images = [img.to(device) for img in images]

        # Model çıktısı al
        outputs = model(images)

        # Çıktıları işle
        for i in range(len(images)):
            # Tahminler (CPU'ya alıyoruz)
            pred_boxes = outputs[i]['boxes'].cpu()
            pred_labels = outputs[i]['labels'].cpu()
            pred_scores = outputs[i]['scores'].cpu()

            # Gerçek Etiketler (Ground Truth)
            gt_boxes = targets[i]['boxes'].cpu()
            gt_labels = targets[i]['labels'].cpu()

            all_preds.append({"boxes": pred_boxes, "labels": pred_labels, "scores": pred_scores})
            all_targets.append({"boxes": gt_boxes, "labels": gt_labels})

# --- 3. METRİK HESAPLAMA FONKSİYONU ---

def compute_metrics(all_preds, all_targets, iou_thresh=0.5):
    TP, FP, FN = 0, 0, 0

    for pred, target in zip(all_preds, all_targets):
        pred_boxes = pred["boxes"]
        pred_scores = pred["scores"]
        gt_boxes = target["boxes"]

        # Tahmin yoksa hepsi False Negative
        if len(pred_boxes) == 0:
            FN += len(gt_boxes)
            continue

        # Gerçek kutu yoksa hepsi False Positive
        if len(gt_boxes) == 0:
            FP += len(pred_boxes)
            continue

        # Tahminleri skora göre sırala (en yüksek skorlu en üstte)
        sorted_indices = torch.argsort(pred_scores, descending=True)
        pred_boxes = pred_boxes[sorted_indices]

        # IoU Hesapla
        ious = box_iou(pred_boxes, gt_boxes)
        matched = torch.zeros(len(gt_boxes), dtype=torch.bool)

        for j in range(len(pred_boxes)):
            if ious.shape[1] > 0:
                # Bu tahmin kutusu en çok hangi gerçek kutuya benziyor?
                iou, idx = torch.max(ious[j], dim=0)

                # IoU eşik değerinden büyükse ve o gerçek kutu daha önce eşleşmediyse -> TP
                if iou >= iou_thresh and not matched[idx]:
                    TP += 1
                    matched[idx] = True
                else:
                    FP += 1
            else:
                FP += 1

        # Eşleşmeyen gerçek kutular -> FN
        FN += (~matched).sum().item()

    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-8)

    return precision, recall, f1

# Hesapla ve Yazdır
precision, recall, f1 = compute_metrics(all_preds, all_targets)

print("\n" + "="*35)
print(f"🚀 TEST SONUÇLARI")
print("="*35)
print(f"Precision (Kesinlik) : {precision:.4f}")
print(f"Recall (Duyarlılık)  : {recall:.4f}")
print(f"F1 Score             : {f1:.4f}")
print("="*35)

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
📂 '/content/MakineGormesi-2/test' klasöründeki 154 görüntü test ediliyor...


Test İlerlemesi: 100%|██████████| 39/39 [00:02<00:00, 13.87it/s]



🚀 TEST SONUÇLARI
Precision (Kesinlik) : 0.0032
Recall (Duyarlılık)  : 0.9030
F1 Score             : 0.0064


In [55]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 35.0 MB/s eta 0:00:00


In [56]:
import torch
import time
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm import tqdm

# Cihaz ayarı
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# --- 1. METRİK TANIMLAMA (COCO Standartları) ---
# class_metrics=True dersen her sınıf için (cam, kagit vs.) ayrı ayrı başarıyı gösterir.
metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True)

print("Test ve Karşılaştırma Metrikleri Hesaplanıyor...")

# --- 2. HIZ TESTİ (FPS) İÇİN BAŞLANGIÇ ---
start_time = time.time()
total_images = 0

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="YOLO Karşılaştırma Testi"):
        images = [img.to(device) for img in images]

        # Ground Truth verilerini hazırla (Dict listesi formatında olmalı)
        target_data = []
        for t in targets:
            target_data.append({
                "boxes": t["boxes"].to(device),
                "labels": t["labels"].to(device)
            })

        # Model Tahmini
        preds = model(images)

        # Metrik kütüphanesine verileri besle
        metric.update(preds, target_data)

        total_images += len(images)

# --- 3. SONUÇLARI HESAPLA ---
end_time = time.time()
inference_time = end_time - start_time
fps = total_images / inference_time

# mAP Sonuçlarını al
results = metric.compute()

# --- 4. RAPORLAMA ---
print("\n" + "="*40)
print(f"📊 SSD vs YOLO KARŞILAŞTIRMA METRİKLERİ")
print("="*40)
print(f"⚡ HIZ (Performans):")
print(f"   - Toplam Süre      : {inference_time:.2f} saniye")
print(f"   - İşlenen Resim    : {total_images}")
print(f"   - FPS (Kare/Saniye): {fps:.2f}")
print("-" * 40)
print(f"🎯 BAŞARI (Accuracy):")
print(f"   - mAP@50 (Genel Başarı)     : {results['map_50'].item():.4f}")
print(f"   - mAP@50-95 (COCO Standardı): {results['map'].item():.4f}")
print("-" * 40)

# Sınıf bazlı sonuçları görmek istersen:
# CLASSES listesi senin kodunun başında tanımlıydı: ['cam', 'kagit', 'metal', 'pil', 'plastik']
# Dikkat: map_per_class sırası label ID'lerine göredir.
if 'map_per_class' in results:
    print("📦 Sınıf Bazlı mAP@50-95 Değerleri:")
    dataset_classes = ['background'] + CLASSES # Senin CLASSES listen
    for i, score in enumerate(results['map_per_class']):
        # i+1 çünkü 0 genelde background olabilir, duruma göre kontrol etmelisin
        class_name = dataset_classes[i] if i < len(dataset_classes) else f"Class {i}"
        print(f"   - {class_name:10s}: {score.item():.4f}")
print("="*40)

Test ve Karşılaştırma Metrikleri Hesaplanıyor...


YOLO Karşılaştırma Testi: 100%|██████████| 39/39 [00:05<00:00,  7.76it/s]



📊 SSD vs YOLO KARŞILAŞTIRMA METRİKLERİ
⚡ HIZ (Performans):
   - Toplam Süre      : 5.04 saniye
   - İşlenen Resim    : 154
   - FPS (Kare/Saniye): 30.58
----------------------------------------
🎯 BAŞARI (Accuracy):
   - mAP@50 (Genel Başarı)     : 0.5650
   - mAP@50-95 (COCO Standardı): 0.2603
----------------------------------------
📦 Sınıf Bazlı mAP@50-95 Değerleri:
   - background: 0.2511
   - cam       : 0.1556
   - kagit     : 0.3615
   - metal     : 0.3778
   - pil       : 0.1554


In [57]:
!zip -r contentssd.zip /content


  adding: content/ (stored 0%)
  adding: content/.config/ (stored 0%)
  adding: content/.config/config_sentinel (stored 0%)
  adding: content/.config/active_config (stored 0%)
  adding: content/.config/default_configs.db (deflated 98%)
  adding: content/.config/configurations/ (stored 0%)
  adding: content/.config/configurations/config_default (deflated 15%)
  adding: content/.config/gce (stored 0%)
  adding: content/.config/.last_survey_prompt.yaml (stored 0%)
  adding: content/.config/.last_opt_in_prompt.yaml (stored 0%)
  adding: content/.config/logs/ (stored 0%)
  adding: content/.config/logs/2025.12.09/ (stored 0%)
  adding: content/.config/logs/2025.12.09/14.41.18.717681.log (deflated 58%)
  adding: content/.config/logs/2025.12.09/14.41.42.675750.log (deflated 57%)
  adding: content/.config/logs/2025.12.09/14.40.47.605300.log (deflated 92%)
  adding: content/.config/logs/2025.12.09/14.41.33.792924.log (deflated 58%)
  adding: content/.config/logs/2025.12.09/14.41.27.893750.log (d

In [60]:
!cp /content/contentssd.zip /content/drive/MyDrive/
